# Sanity check: hex connectivity spatial consistency

Verifies that connected hex pairs are geographically plausible.

**Method:** For a random sample of source hexes, compute the great-circle distance
from each source to its connected targets (weighted by connectivity weight).
If hex ordering is correct, the weight-averaged target centroid should be close
to the source. If scrambled, distances will be implausibly large.

**Expected ranges** (larval dispersal, surface currents):
- 0–7 days: median weighted distance < 500 km
- 7–28 days: median weighted distance < 1500 km

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Parameters
N_SAMPLE = 200       # number of random source hexes to check
SEED     = 42
MAX_OK_MEDIAN_KM = 1500  # flag if median weighted distance exceeds this

_cwd = Path.cwd()
if (_cwd / "../database/data").exists():
    DATA_DIR = (_cwd / "../database/data").resolve()
else:
    DATA_DIR = (_cwd / "../../database/data").resolve()

print(f"DATA_DIR: {DATA_DIR}")

In [ ]:
# Load hex lon/lat from meta.json
meta = json.loads((DATA_DIR / "meta.json").read_text())
hex_ids = np.array([int(k) for k in meta["id"].values()])
lons = np.array([meta["lon"][str(i)] for i in hex_ids])
lats = np.array([meta["lat"][str(i)] for i in hex_ids])
id_to_lon = dict(zip(hex_ids, lons))
id_to_lat = dict(zip(hex_ids, lats))
print(f"Loaded {len(hex_ids)} hexes")

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    lo1, la1, lo2, la2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlo = lo2 - lo1
    dla = la2 - la1
    a = np.sin(dla/2)**2 + np.cos(la1) * np.cos(la2) * np.sin(dlo/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def check_pq(path, label):
    df = pd.read_parquet(path)
    df["end_id"] = df["end_id"].astype(int)

    src_ids = df["start_id"].unique()
    rng = np.random.default_rng(SEED)
    sample = rng.choice(src_ids, size=min(N_SAMPLE, len(src_ids)), replace=False)

    weighted_dists = []
    for src in sample:
        if src not in id_to_lon:
            continue
        rows = df[df["start_id"] == src]
        tgt_lons = np.array([id_to_lon.get(t, np.nan) for t in rows["end_id"]])
        tgt_lats = np.array([id_to_lat.get(t, np.nan) for t in rows["end_id"]])
        weights  = rows["weight"].values
        valid = np.isfinite(tgt_lons) & np.isfinite(tgt_lats)
        if valid.sum() == 0:
            continue
        dists = haversine_km(id_to_lon[src], id_to_lat[src], tgt_lons[valid], tgt_lats[valid])
        w = weights[valid]
        weighted_dists.append(np.average(dists, weights=w))

    arr = np.array(weighted_dists)
    med = np.median(arr)
    p95 = np.percentile(arr, 95)
    flag = "FAIL" if med > MAX_OK_MEDIAN_KM else "OK"
    print(f"{label}: n={len(arr)}  median={med:.0f} km  p95={p95:.0f} km  [{flag}]")
    return arr

In [ ]:
results = {}
for pq in sorted(DATA_DIR.glob("connectivity_*.pq")):
    label = pq.stem.replace("connectivity_", "")
    results[label] = check_pq(pq, label)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
labels = list(results.keys())
medians = [np.median(v) for v in results.values()]
p95s    = [np.percentile(v, 95) for v in results.values()]

x = np.arange(len(labels))
ax.bar(x, medians, label="median", alpha=0.8)
ax.bar(x, p95s, label="p95", alpha=0.4)
ax.axhline(MAX_OK_MEDIAN_KM, color="red", linestyle="--", label=f"threshold ({MAX_OK_MEDIAN_KM} km)")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("weighted mean distance (km)")
ax.set_title("Connectivity spatial sanity check")
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / "sanity_check_distances.png", dpi=150)
plt.show()
print("Done.")